<a href="https://colab.research.google.com/github/asierra383/ScamBusters_Agent/blob/main/Scam_Busters_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

START

In [13]:
pip install google-adk

In [14]:
#Import ADK components
import gspread
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types
import google.genai as genai

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [15]:
import os
from google.colab import userdata
api_key = userdata.get('GOOGLE_API_KEY')
os.environ['GOOGLE_API_KEY'] = api_key

In [16]:
#Configure Retry options
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

In [17]:
# 1. Finds the main HYIP monitor sites
discovery_agent = LlmAgent(
    name="DiscoveryAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        api_key=api_key,
        retry_options=retry_config
    ),
    instruction="Find current HYIP monitoring websites. List only the URLs.",
    tools=[google_search],
    output_key="hyip_monitor_list"
)

# 2. Next agent to browse the sites advertised in HYIP monitoring websites and extract content
browser_agent = LlmAgent(
    name="BrowserAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        api_key=api_key,
        retry_options=retry_config),
    instruction="""Given the list of HYIP monitoring websites in {hyip_monitor_list},
    use Google Search to visit each website and extract the cryptocurrency websites it has linked.
    Combine all extracted websites into a single comprehensive string and make
    this combined content available as 'site_content' for the next agent.""",
    tools=[google_search],
    output_key="site_content"
)

print("✅ Discovery Agent and Browser Agent defined.")

✅ Discovery Agent and Browser Agent defined.


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
# --- STEP 1: SPREADSHEET SETUP ---
import gspread
gc = gspread.service_account(filename='/content/drive/My Drive/Colab Notebooks/scambusters.json')
sh = gc.open("ScamBusters_Tracker")
worksheet = sh.get_worksheet(0)

# --- STEP 2: DEFINE THE TOOL AS A FUNCTION ---
def add_to_spreadsheet(url: str, category: str, notes: str):
    """
    Appends a new website's details to the tracking spreadsheet.
    Use this whenever a new relevant website is discovered.

    Args:
        url (str): The URL of the website to add.
        category (str): The category for the website (e.g., 'HYIP Monitor').
        notes (str): Any additional notes about the website.
    """
    try:
        # Check for duplicates before adding
        existing_urls = worksheet.col_values(1)
        if url in existing_urls:
            return f"Skipped: {url} is already in the sheet."

        worksheet.append_row([url, category, notes])
        return f"Successfully added {url} to the spreadsheet."
    except Exception as e:
        return f"Error updating spreadsheet: {str(e)}"

print("✅ Spreadsheet function defined.")

✅ Spreadsheet function defined.


In [20]:
# 3. Processes site content and logs new URLs to the spreadsheet
spreadsheet_agent = LlmAgent(
    name="SpreadsheetAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        api_key=api_key,
        retry_options=retry_config
    ),
    instruction="""Analyze the content in {site_content}.
    Identify new, unique HYIP related URLs mentioned in the content.
    For each new unique URL found, use the 'add_to_spreadsheet' tool with category 'HYIP Monitor' and notes 'Discovered by agent'.
    Pass the original {site_content} along to the next agent unchanged.""",
    tools=[add_to_spreadsheet],
    output_key="site_content"
)

print("✅ Spreadsheet Agent defined.")

✅ Spreadsheet Agent defined.


In [21]:
# 4. Analyzes the extracted text for your specific scam keywords
analyst_agent = LlmAgent(
    name="AnalystAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
    retry_options=retry_config),
    instruction="""Analyze the content in {site_content}.
    Search for scam markers like 'guaranteed high returns' or anonymous teams.
    Provide a final risk assessment.""",
    output_key="final_scam_report"
)

print("✅ Analyst Agent defined.")

✅ Analyst Agent defined.


In [22]:
# Re-initialize the pipeline with updated agents
scam_pipeline = SequentialAgent(
    name="ScamDetectionPipeline",
    sub_agents=[discovery_agent, browser_agent, spreadsheet_agent, analyst_agent]
)

print("✅ Pipeline updated with agents.")

✅ Pipeline updated with agents.


In [23]:
#Run the multi-agent system
runner = InMemoryRunner(agent=scam_pipeline)

print("✅ Runner updated.")

✅ Runner updated.


In [24]:
#Run by prompting for answer
response = await runner.run_debug(
    "What hyip monitoring websites have recently updated their lists within the past day?"
)


 ### Created new session: debug_session_id

User > What hyip monitoring websites have recently updated their lists within the past day?
DiscoveryAgent > The following HYIP monitoring websites appear to have updated their lists within the past day, as indicated by recent activity or listed payouts/reviews:

*   **Allmonitors24.com**: Shows recent reviews and additions from April 20, 2026.
*   **HYIP.BIZ**: Lists a last update of April 19, 2026, and mentions payouts from "9 hours ago".
*   **CryptoHyip.net**: Lists "Last SCAM Projects" with dates of April 19 and April 18, 2026, and has recent paying HYIPs listed with start dates in April 2026.
*   **upayhyip.com**: Shows recent news and lists for April 2026, with new listings and top profitable programs updated.
*   **All HYIP Monitors .com**: Lists recently added and fastest growing programs, with many having dates or activity indicating recent updates.
BrowserAgent > Here are the cryptocurrency websites linked by the HYIP monitoring w